# L11 · ACT 与 SmolVLA 策略训练

本 notebook 把讲义中的证据阶梯落实为可执行流程：检查一个真实样本，读取当前安装版本的策略默认值，审计两条最终命令，可选地让每种策略在 GPU 上真实运行一步，检查生成的 checkpoint，并用同一个样本重新加载两种策略。

完整实验需要 GPU。默认路径会关闭两次 smoke，避免仅仅打开或从头运行 notebook 就意外启动高成本任务。dry-run 只能证明命令组装正确；本 notebook 中的任何结果都不是闭环任务证据。

## 运行前准备

先安装完整课程环境；若使用 AMD 平台，还需按照 COMPATIBILITY.md 的说明，用已经验证的 ROCm wheels 替换跨平台 PyTorch 包。请直接从这个环境启动 Jupyter，不要向 Python 注入额外路径。

启动 kernel 前设置以下环境变量：

- RG101_REPO_ID：LeRobot 数据集的逻辑 ID，默认为 genesis/fruit_pick。
- RG101_DATASET_ROOT：本地数据集目录，默认为 datasets/fruit_pick。
- RG101_OUTPUT_ROOT：生成的运行目录，默认为 outputs/train/l11。
- RG101_SEED：需要记录的 seed，默认为 1000。
- RG101_RUN_SMOKE：进行数据和命令检查时保持为 0；只有准备执行两次真实 GPU smoke 时才设为 1。
- RG101_SMOLVLA_BASE_SNAPSHOT 和 RG101_SMOLVLA_VLM_SNAPSHOT：两个已验证 commit 对应的 Hugging Face 本地 snapshot 精确目录。要么同时设置，要么都不设置。

启动 kernel 前还应选好目标 GPU。在已验证的 AMD 系统上，PyTorch 通过 torch.cuda API 暴露 ROCm，因此 LeRobot 的 device 字符串仍为 cuda。回退到 CPU 不能算作 smoke 通过。

In [ ]:
import importlib.metadata as package_metadata
import json
import os
import shlex
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import torch
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.paths import DATASETS_DIR, PATHS, TRAIN_OUTPUTS_DIR
from robo_genesis.training_contract import (
    SMOLVLA_BASE_REPO_ID,
    SMOLVLA_BASE_REVISION,
    SMOLVLA_CAMERA_RENAME,
    SMOLVLA_VLM_REPO_ID,
    SMOLVLA_VLM_REVISION,
    audit_checkpoint,
    audit_smolvla_snapshots,
    command_options,
    horizon_evidence,
    parse_training_metrics,
    resolve_numeric_checkpoint,
)


def environment_flag(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    normalized = value.strip().lower()
    if normalized not in {"0", "1", "false", "true", "no", "yes"}:
        raise ValueError(f"{name} must be a boolean value, got {value!r}")
    return normalized in {"1", "true", "yes"}


def installed_version(distribution):
    try:
        return package_metadata.version(distribution)
    except package_metadata.PackageNotFoundError:
        return "not-installed"


lesson = load_course_manifest().lesson("L11")
assert lesson.status.value == "gpu-verified"

repo_id = os.environ.get("RG101_REPO_ID", "genesis/fruit_pick")
dataset_root = Path(
    os.environ.get("RG101_DATASET_ROOT", str(DATASETS_DIR / "fruit_pick"))
).expanduser().resolve()
output_root = Path(
    os.environ.get("RG101_OUTPUT_ROOT", str(TRAIN_OUTPUTS_DIR / "l11"))
).expanduser().resolve()
seed = int(os.environ.get("RG101_SEED", "1000"))
run_smoke = environment_flag("RG101_RUN_SMOKE", default=False)

base_snapshot_value = os.environ.get("RG101_SMOLVLA_BASE_SNAPSHOT")
vlm_snapshot_value = os.environ.get("RG101_SMOLVLA_VLM_SNAPSHOT")
base_snapshot = Path(base_snapshot_value).expanduser().resolve() if base_snapshot_value else None
vlm_snapshot = Path(vlm_snapshot_value).expanduser().resolve() if vlm_snapshot_value else None

output_root.mkdir(parents=True, exist_ok=True)
cuda_available = torch.cuda.is_available()
visible_devices = torch.cuda.device_count()
actual_device = torch.cuda.get_device_name(0) if cuda_available else None
runtime_evidence = {
    "python": sys.version.split()[0],
    "lerobot": installed_version("lerobot"),
    "torch": torch.__version__,
    "torch_hip": torch.version.hip,
    "torch_cuda": torch.version.cuda,
    "cuda_api_available": cuda_available,
    "visible_devices": visible_devices,
    "actual_device": actual_device,
    "repo_id": repo_id,
    "dataset_root": str(dataset_root),
    "output_root": str(output_root),
    "seed": seed,
    "run_smoke": run_smoke,
    "hf_hub_offline": os.environ.get("HF_HUB_OFFLINE"),
    "transformers_offline": os.environ.get("TRANSFORMERS_OFFLINE"),
}
print(json.dumps(runtime_evidence, indent=2, ensure_ascii=False))

if run_smoke and not cuda_available:
    raise RuntimeError(
        "RG101_RUN_SMOKE=1 requires an actual GPU. CPU fallback is not accepted for this lab."
    )

## 1. 在数据边界尽早失败

第一个可执行门禁会在分配模型前读取元数据和一个真实解码样本，检查共同的 9 维 state/action 关节顺序、两路 RGB、task 文本、FPS，以及 episode/frame 数量。数据集缺失或格式不符时，代码会给出可操作的错误，而不会用随机 tensor 替代。

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

from robo_genesis.record_dataset import JOINT_NAMES

info_path = dataset_root / "meta" / "info.json"
if not info_path.is_file():
    raise FileNotFoundError(
        f"Dataset metadata is missing: {info_path}. "
        "Set RG101_DATASET_ROOT to the local LeRobot dataset produced in the earlier lessons."
    )

metadata = LeRobotDatasetMetadata(repo_id, root=dataset_root)
dataset = LeRobotDataset(repo_id, root=dataset_root, video_backend="pyav")
if len(dataset) < 1:
    raise ValueError(f"Dataset contains no readable samples: {dataset_root}")
sample = dataset[0]

STATE_KEY = "observation.state"
ACTION_KEY = "action"
IMAGE_KEYS = (
    "observation.images.world",
    "observation.images.wrist",
)


def as_numpy(value):
    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    return np.asarray(value)


def check_vector(key):
    feature = metadata.features.get(key)
    if feature is None:
        raise KeyError(f"Dataset metadata is missing {key}")
    if tuple(feature["shape"]) != (9,) or feature["dtype"] != "float32":
        raise ValueError(f"{key} metadata must be shape (9,) float32, got {feature}")
    if tuple(feature.get("names") or ()) != tuple(JOINT_NAMES):
        raise ValueError(f"{key} joint names do not match the course joint order")

    value = as_numpy(sample[key])
    if value.shape != (9,) or value.dtype != np.float32:
        raise ValueError(f"{key} sample must be shape (9,) float32, got {value.shape} {value.dtype}")
    if not np.isfinite(value).all():
        raise ValueError(f"{key} sample contains non-finite values")
    return value


state = check_vector(STATE_KEY)
action = check_vector(ACTION_KEY)
decoded_shapes = {}
for key in IMAGE_KEYS:
    feature = metadata.features.get(key)
    if feature is None:
        raise KeyError(f"Dataset metadata is missing {key}")
    stored_h, stored_w, stored_c = tuple(feature["shape"])
    if stored_c != 3:
        raise ValueError(f"{key} metadata must describe RGB HWC frames, got {feature['shape']}")

    image = as_numpy(sample[key])
    if image.shape != (3, stored_h, stored_w):
        raise ValueError(
            f"{key} decoded sample must be CHW {(3, stored_h, stored_w)}, got {image.shape}"
        )
    if not np.isfinite(image).all():
        raise ValueError(f"{key} decoded sample contains non-finite values")
    decoded_shapes[key] = tuple(image.shape)

if decoded_shapes[IMAGE_KEYS[0]] != decoded_shapes[IMAGE_KEYS[1]]:
    raise ValueError(f"Camera shapes differ: {decoded_shapes}")

task_text = sample.get("task")
if not isinstance(task_text, str) or not task_text.strip():
    raise ValueError("The real dataset sample must resolve to non-empty task text")
if metadata.fps <= 0 or metadata.total_episodes <= 0 or metadata.total_frames <= 0:
    raise ValueError("Dataset FPS, episode count, and frame count must all be positive")

dataset_evidence = {
    "status": "PASS",
    "repo_id": repo_id,
    "root": str(metadata.root),
    "episodes": metadata.total_episodes,
    "frames": metadata.total_frames,
    "fps": metadata.fps,
    "state": {"shape": state.shape, "dtype": str(state.dtype), "finite": True},
    "action": {"shape": action.shape, "dtype": str(action.dtype), "finite": True},
    "decoded_images": decoded_shapes,
    "task": task_text,
}
print(json.dumps(dataset_evidence, indent=2, ensure_ascii=False, default=list))

## 2. 把当前配置与时间联系起来

从当前安装的 LeRobot 版本读取策略默认值，不在 notebook 中手抄。预测时域等于 chunk_size 除以数据集 FPS；重新规划间隔等于 n_action_steps 除以 FPS。虽然两种策略目前的默认配置分别让二者相等，但它们表示不同含义。

缩小后的 ACT 明确只用于验证管线，不能把它当成一个小型 baseline，与 SmolVLA 比较任务效果。

In [ ]:
act_default = ACTConfig()
smolvla_default = SmolVLAConfig()

configuration_evidence = {
    "act_default": {
        "horizon": horizon_evidence(
            chunk_size=act_default.chunk_size,
            n_action_steps=act_default.n_action_steps,
            fps=metadata.fps,
        ),
        "pretrained_backbone_weights": act_default.pretrained_backbone_weights,
    },
    "act_pipeline_only_smoke": horizon_evidence(
        chunk_size=10,
        n_action_steps=10,
        fps=metadata.fps,
    ),
    "smolvla_default": {
        "horizon": horizon_evidence(
            chunk_size=smolvla_default.chunk_size,
            n_action_steps=smolvla_default.n_action_steps,
            fps=metadata.fps,
        ),
        "vlm_model_name": smolvla_default.vlm_model_name,
        "freeze_vision_encoder": smolvla_default.freeze_vision_encoder,
        "train_expert_only": smolvla_default.train_expert_only,
        "train_state_proj": smolvla_default.train_state_proj,
    },
}
print(json.dumps(configuration_evidence, indent=2, ensure_ascii=False, default=str))

## 3. 审计 SmolVLA 的两个模型 revision

只有本地 base snapshot 还不能固定 SmolVLA 随后加载的 VLM。因此，这个门禁会同时核对两个完整的 40 位 revision、必要文件、base 策略类型，以及其配置中声明的 VLM 仓库。

Hugging Face cache 中精确的 snapshots/commit 路径可以直接证明 revision。复制到其他位置的 snapshot 也可以提供一份 robo_genesis_snapshot.json 来源记录，其中写明 repo_id 和 revision。审计只读取本地文件，不会下载模型。

In [ ]:
if (base_snapshot is None) != (vlm_snapshot is None):
    raise ValueError(
        "Set both RG101_SMOLVLA_BASE_SNAPSHOT and RG101_SMOLVLA_VLM_SNAPSHOT, or neither."
    )

snapshot_evidence = None
if base_snapshot is not None and vlm_snapshot is not None:
    snapshot_evidence = audit_smolvla_snapshots(base_snapshot, vlm_snapshot)
    print("PASS — pinned SmolVLA base and VLM snapshots")
    print(json.dumps(snapshot_evidence.as_dict(), indent=2, ensure_ascii=False))
else:
    print("NOT READY — pinned SmolVLA model content was not supplied")
    print(f"expected base: {SMOLVLA_BASE_REPO_ID}@{SMOLVLA_BASE_REVISION}")
    print(f"expected VLM:  {SMOLVLA_VLM_REPO_ID}@{SMOLVLA_VLM_REVISION}")

if run_smoke and snapshot_evidence is None:
    raise RuntimeError(
        "RG101_RUN_SMOKE=1 requires both audited local SmolVLA snapshot paths."
    )

## 4. 生成并审计两条 dry-run 命令

下面通过 --dry-run 调用项目 wrapper，不会打开数据集、加载模型、分配 GPU tensor 或写入 checkpoint。代码会解析 wrapper 输出的可安全还原 shell 命令，并核对路径、job name、默认 batch、device、PyAV、关闭外部发布、策略选择器和 SmolVLA 相机映射。

如果已经提供固定 snapshot，SmolVLA 命令还会带上审计后的本地 VLM 路径；否则 wrapper 会明确报告 UNPINNED。后一种命令仍可用于了解接口，但尚不能用于可复现训练。

In [ ]:
def run_wrapper(command):
    print("$ " + shlex.join(command))
    started = time.perf_counter()
    completed = subprocess.run(
        command,
        cwd=PATHS.project_root,
        text=True,
        capture_output=True,
        check=False,
    )
    elapsed = time.perf_counter() - started
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with return code {completed.returncode}")
    return {"completed": completed, "elapsed_seconds": elapsed}


def printed_trainer_command(stdout):
    lines = [line for line in stdout.splitlines() if line.startswith("[train] ")]
    if len(lines) != 1:
        raise ValueError(f"Expected one [train] command line, found {len(lines)}")
    return shlex.split(lines[0].removeprefix("[train] "))


def wrapper_command(policy, name, *, batch_size=None, dry_run=False):
    command = [
        sys.executable,
        "-m",
        "robo_genesis.train_policy",
        policy,
        "--repo-id",
        repo_id,
        "--dataset-root",
        str(dataset_root),
        "--name",
        name,
        "--output-dir",
        str(output_root / name),
        "--steps",
        "1",
        "--save-freq",
        "1",
        "--log-freq",
        "1",
        "--num-workers",
        "0",
        "--seed",
        str(seed),
        "--device",
        "cuda",
        "--video-backend",
        "pyav",
    ]
    if batch_size is not None:
        command.extend(["--batch-size", str(batch_size)])
    if policy == "smolvla" and snapshot_evidence is not None:
        command.extend(
            [
                "--policy-path",
                str(snapshot_evidence.base.path),
                "--smolvla-vlm-path",
                str(snapshot_evidence.vlm.path),
            ]
        )
    if dry_run:
        command.append("--dry-run")
    return command


def check_common_options(options, name, expected_batch):
    expected = {
        "--dataset.repo_id": repo_id,
        "--dataset.root": str(dataset_root),
        "--output_dir": str((output_root / name).resolve()),
        "--job_name": name,
        "--batch_size": str(expected_batch),
        "--steps": "1",
        "--save_freq": "1",
        "--log_freq": "1",
        "--num_workers": "0",
        "--seed": str(seed),
        "--policy.device": "cuda",
        "--policy.push_to_hub": "false",
        "--wandb.enable": "false",
        "--dataset.video_backend": "pyav",
    }
    for option, expected_value in expected.items():
        if options.get(option) != expected_value:
            raise AssertionError(f"{option}: {options.get(option)!r} != {expected_value!r}")


act_dry_name = "l11-act-dry-run"
act_dry_result = run_wrapper(wrapper_command("act", act_dry_name, dry_run=True))
act_trainer_command = printed_trainer_command(act_dry_result["completed"].stdout)
act_options = command_options(act_trainer_command)
check_common_options(act_options, act_dry_name, expected_batch=8)
assert act_options["--policy.type"] == "act"
assert "--policy.path" not in act_options
assert "--rename_map" not in act_options

smolvla_dry_name = "l11-smolvla-dry-run"
smolvla_dry_result = run_wrapper(wrapper_command("smolvla", smolvla_dry_name, dry_run=True))
smolvla_trainer_command = printed_trainer_command(smolvla_dry_result["completed"].stdout)
smolvla_options = command_options(smolvla_trainer_command)
check_common_options(smolvla_options, smolvla_dry_name, expected_batch=4)
expected_smolvla_path = (
    str(snapshot_evidence.base.path)
    if snapshot_evidence is not None
    else SMOLVLA_BASE_REPO_ID
)
assert smolvla_options["--policy.path"] == expected_smolvla_path
assert json.loads(str(smolvla_options["--rename_map"])) == SMOLVLA_CAMERA_RENAME
if snapshot_evidence is None:
    assert "--policy.vlm_model_name" not in smolvla_options
else:
    assert smolvla_options["--policy.vlm_model_name"] == str(snapshot_evidence.vlm.path)

print("PASS — ACT and SmolVLA command contracts; no dataset, model, or GPU was opened")

## 5. 准备两次单步 GPU smoke

ACT smoke 会关闭 ImageNet 下载并缩小 CVAE/Transformer，仅用于验证 decode、前向、loss、反向、更新和保存链路。SmolVLA 保留已验证的 base 架构和微调边界，同时使用两个固定的本地 snapshot。

下面的 preflight 仍然是 dry-run。只有在启动 kernel 前设置了 RG101_RUN_SMOKE=1，随后的单元格才会真正开始训练。

In [ ]:
act_smoke_command = wrapper_command("act", "l11-act-smoke", batch_size=1)
act_smoke_command.extend(
    [
        "--",
        "--policy.pretrained_backbone_weights=null",
        "--policy.chunk_size=10",
        "--policy.n_action_steps=10",
        "--policy.dim_model=64",
        "--policy.n_heads=4",
        "--policy.dim_feedforward=128",
        "--policy.n_encoder_layers=1",
        "--policy.n_decoder_layers=1",
        "--policy.n_vae_encoder_layers=1",
        "--policy.latent_dim=8",
    ]
)

smolvla_smoke_command = (
    wrapper_command("smolvla", "l11-smolvla-smoke", batch_size=1)
    if snapshot_evidence is not None
    else None
)


def as_dry_run(command):
    result = list(command)
    separator = result.index("--") if "--" in result else len(result)
    result.insert(separator, "--dry-run")
    return result


act_preflight = run_wrapper(as_dry_run(act_smoke_command))
act_smoke_options = command_options(
    printed_trainer_command(act_preflight["completed"].stdout)
)
assert act_smoke_options["--policy.pretrained_backbone_weights"] == "null"
assert act_smoke_options["--policy.chunk_size"] == "10"
assert act_smoke_options["--policy.n_action_steps"] == "10"
assert act_smoke_options["--policy.dim_model"] == "64"
print("PASS — ACT command is a pipeline-only smoke configuration")

if smolvla_smoke_command is None:
    print("SKIP — SmolVLA smoke preflight requires the two pinned local snapshots")
else:
    smolvla_preflight = run_wrapper(as_dry_run(smolvla_smoke_command))
    smolvla_smoke_options = command_options(
        printed_trainer_command(smolvla_preflight["completed"].stdout)
    )
    assert smolvla_smoke_options["--policy.path"] == str(snapshot_evidence.base.path)
    assert smolvla_smoke_options["--policy.vlm_model_name"] == str(snapshot_evidence.vlm.path)
    assert json.loads(str(smolvla_smoke_options["--rename_map"])) == SMOLVLA_CAMERA_RENAME
    print("PASS — SmolVLA smoke command uses both pinned snapshots and the camera map")

In [ ]:
smoke_processes = {}
if run_smoke:
    assert smolvla_smoke_command is not None
    smoke_processes["act"] = run_wrapper(act_smoke_command)
    smoke_processes["smolvla"] = run_wrapper(smolvla_smoke_command)
else:
    print("SKIP — GPU smoke runs are opt-in; set RG101_RUN_SMOKE=1 before starting the kernel")

## 6. 检查有限指标和 checkpoint 软件包

子进程成功退出还不够。对每次实际执行的 smoke，都必须找到同时包含有限 loss 与 gradient norm 的日志记录，确认 checkpoints/last 指向最新的数字 step，并检查非空权重、策略配置、训练配置、pre/postprocessors 和其中引用的 processor state 文件。

两种 loss 的目标和尺度不同，不能把它们横向比较后用来排名策略，也不能从一个点画出趋势。

In [ ]:
checkpoint_evidence = {}
if run_smoke:
    for policy_name, process in smoke_processes.items():
        combined_log = process["completed"].stdout + "\n" + process["completed"].stderr
        metrics = parse_training_metrics(combined_log)
        run_directory = output_root / f"l11-{policy_name}-smoke"
        model_directory = resolve_numeric_checkpoint(run_directory)
        artifact = audit_checkpoint(
            model_directory,
            expected_policy_type=policy_name,
            expected_action_dim=9,
        )
        if artifact["dataset_repo_id"] != repo_id:
            raise AssertionError(f"{policy_name} checkpoint records the wrong dataset")
        if artifact["seed"] != seed:
            raise AssertionError(f"{policy_name} checkpoint records the wrong seed")
        artifact["metrics"] = metrics
        artifact["elapsed_seconds_this_run"] = process["elapsed_seconds"]
        checkpoint_evidence[policy_name] = artifact
    print(json.dumps(checkpoint_evidence, indent=2, ensure_ascii=False, default=str))
else:
    print("SKIP — no checkpoint is claimed because GPU smoke was not run")

## 7. 用同一个真实样本重新加载两个 checkpoint

LeRobot 会把视频样本解码成通道优先的浮点 tensor，而项目推理 helper 会先接收原始 HWC uint8 图像，再应用保存的预处理器。下面会明确展示并检查这一步转换。

每个策略都必须返回一个数值有限的 9 维 float32 动作。这只是开环单样本探针：它没有构建 Genesis、没有应用动作，也没有评估任务是否成功。

In [ ]:
def decoded_rgb_to_hwc_uint8(value):
    image = as_numpy(value)
    if image.ndim != 3:
        raise ValueError(f"Decoded image must have three dimensions, got {image.shape}")
    if image.shape[0] in (3, 4):
        image = np.moveaxis(image, 0, -1)
    if image.shape[-1] == 4:
        image = image[..., :3]
    if image.shape[-1] != 3 or not np.isfinite(image).all():
        raise ValueError(f"Decoded image is not finite RGB data: {image.shape}")
    if np.issubdtype(image.dtype, np.floating):
        if image.min() < 0.0 or image.max() > 1.0 + 1e-6:
            raise ValueError("Floating decoded images must be in the [0, 1] range")
        image = np.rint(np.clip(image, 0.0, 1.0) * 255.0).astype(np.uint8)
    elif image.dtype != np.uint8:
        raise ValueError(f"Unsupported decoded image dtype: {image.dtype}")
    return np.ascontiguousarray(image)


raw_observation = {
    STATE_KEY: np.ascontiguousarray(state, dtype=np.float32),
    **{key: decoded_rgb_to_hwc_uint8(sample[key]) for key in IMAGE_KEYS},
}
assert all(raw_observation[key].shape[-1] == 3 for key in IMAGE_KEYS)

reload_evidence = {}
if run_smoke:
    import gc

    from robo_genesis.eval_policy import load_policy

    for policy_name in ("act", "smolvla"):
        bundle = load_policy(
            checkpoint_evidence[policy_name]["path"],
            repo_id,
            str(dataset_root),
            "cuda",
        )
        bundle.reset()
        predicted_action = bundle.select_action(raw_observation, task_text)
        if predicted_action.shape != (9,):
            raise AssertionError(f"{policy_name} action shape is {predicted_action.shape}, not (9,)")
        if predicted_action.dtype != np.float32:
            raise AssertionError(f"{policy_name} action dtype is {predicted_action.dtype}, not float32")
        if not np.isfinite(predicted_action).all():
            raise AssertionError(f"{policy_name} action contains non-finite values")
        reload_evidence[policy_name] = {
            "policy_type": bundle.policy_type,
            "actual_device": str(bundle.device),
            "raw_image_keys": list(IMAGE_KEYS),
            "task_text_present": bool(task_text.strip()),
            "action_shape": predicted_action.shape,
            "action_dtype": str(predicted_action.dtype),
            "action_finite": True,
        }
        del bundle
        gc.collect()
        torch.cuda.empty_cache()
    print(json.dumps(reload_evidence, indent=2, ensure_ascii=False, default=list))
else:
    print("SKIP — open-loop checkpoint reload requires the two completed smoke runs")

## 8. 先设计完整运行，不直接启动

长时间训练是需要主动选择的课后实验。只有先记录数据集版本和规模、模型 revisions、按秒计算的时域、优化预算、资源、保存/日志频率以及交给 L12 的评估方案，才能替换所有 CHOOSE 值。

模板在填写前有意保持不可执行。运行这个单元格只会打印命令和明确的 skip，不会启动任何训练任务。

In [ ]:
base_for_template = (
    str(snapshot_evidence.base.path)
    if snapshot_evidence is not None
    else "PINNED_SMOLVLA_BASE_SNAPSHOT"
)
vlm_for_template = (
    str(snapshot_evidence.vlm.path)
    if snapshot_evidence is not None
    else "PINNED_SMOLVLA_VLM_SNAPSHOT"
)

act_full_command = [
    sys.executable,
    "-m",
    "robo_genesis.train_policy",
    "act",
    "--repo-id",
    repo_id,
    "--dataset-root",
    str(dataset_root),
    "--name",
    "act-fruit-pick",
    "--output-dir",
    str(output_root / "act-fruit-pick"),
    "--steps",
    "CHOOSE_INTEGER",
    "--batch-size",
    "CHOOSE_INTEGER",
    "--save-freq",
    "CHOOSE_INTEGER",
    "--log-freq",
    "CHOOSE_INTEGER",
    "--num-workers",
    "CHOOSE_INTEGER",
    "--seed",
    "CHOOSE_INTEGER",
    "--device",
    "cuda",
    "--video-backend",
    "pyav",
    "--",
    "--policy.pretrained_backbone_weights=CHOOSE_IMAGENET_IDENTIFIER_OR_NULL",
    "--policy.chunk_size=CHOOSE_INTEGER",
    "--policy.n_action_steps=CHOOSE_INTEGER",
    "--policy.optimizer_lr=CHOOSE_FLOAT",
]

smolvla_full_command = [
    sys.executable,
    "-m",
    "robo_genesis.train_policy",
    "smolvla",
    "--repo-id",
    repo_id,
    "--dataset-root",
    str(dataset_root),
    "--policy-path",
    base_for_template,
    "--smolvla-vlm-path",
    vlm_for_template,
    "--name",
    "smolvla-fruit-pick",
    "--output-dir",
    str(output_root / "smolvla-fruit-pick"),
    "--steps",
    "CHOOSE_INTEGER",
    "--batch-size",
    "CHOOSE_INTEGER",
    "--save-freq",
    "CHOOSE_INTEGER",
    "--log-freq",
    "CHOOSE_INTEGER",
    "--num-workers",
    "CHOOSE_INTEGER",
    "--seed",
    "CHOOSE_INTEGER",
    "--device",
    "cuda",
    "--video-backend",
    "pyav",
    "--",
    "--policy.chunk_size=CHOOSE_INTEGER",
    "--policy.n_action_steps=CHOOSE_INTEGER",
    "--policy.optimizer_lr=CHOOSE_FLOAT",
    "--policy.freeze_vision_encoder=CHOOSE_TRUE_OR_FALSE",
    "--policy.train_expert_only=CHOOSE_TRUE_OR_FALSE",
    "--policy.train_state_proj=CHOOSE_TRUE_OR_FALSE",
]

print("SKIP: full training is opt-in; replace every CHOOSE value from a reviewed run record")
print("ACT template:")
print(shlex.join(act_full_command))
print("SmolVLA template:")
print(shlex.join(smolvla_full_command))

## 9. 反思证据等级

根据上面生成的记录回答：

1. 哪些检查在分配任何策略之前就已经通过？
2. 如果两次 dry-run 通过，但 RG101_RUN_SMOKE 仍为关闭状态，还有哪些内容没有验证？
3. 即使单步 loss 与 gradient norm 都是有限值，为什么仍然不能判断收敛？
4. 哪些 checkpoint 文件分别保存数据身份、策略配置、processors 和 SmolVLA 相机映射？
5. 为什么一次数值有限的开环动作仍然不能说明抓取成功？
6. 必须把哪个精确产物和哪套协议交给 L12？

没有执行的工作请写明“未运行”，不要用另一台机器或更早兼容性测试中记得的数值填补空缺。

In [ ]:
evidence_summary = {
    "dataset_gate": "PASS — one real sample and metadata checked",
    "runtime_config": "PASS — policy defaults read from installed LeRobot",
    "dry_run": "PASS — command assembly only",
    "smolvla_revisions": (
        "PASS — both local snapshots audited"
        if snapshot_evidence is not None
        else "NOT READY — no pinned local snapshots supplied"
    ),
    "gpu_smoke": (
        "PASS — ACT and SmolVLA each completed one step"
        if run_smoke
        else "NOT RUN — RG101_RUN_SMOKE was not enabled"
    ),
    "checkpoint_reload": (
        "PASS — two open-loop single-sample probes"
        if run_smoke
        else "NOT RUN — requires completed smoke checkpoints"
    ),
    "full_training": "NOT RUN — opt-in take-home experiment",
    "closed_loop_evaluation": "NOT RUN — belongs to L12",
}
print(json.dumps(evidence_summary, indent=2, ensure_ascii=False))

## 衔接 L12

L11 最终交付的是带版本、可检查的 checkpoint 产物，以及一份准确说明哪些工作已经运行、哪些没有运行的证据记录。L12 会把这样的产物放进 Genesis 控制循环，随时间执行动作，应用带 seed 的任务判据，并报告闭环结果。不要让本 notebook 的证据越过这条边界。